In [0]:
%sql
CREATE TABLE IF NOT EXISTS mba.trusted.d_estacao_meterologica(
    id_estacao BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Chave substituta do fato clima diário',
    codigo_wmo STRING COMMENT 'Código OMM/WMO da estação meteorológica',
    estacao STRING COMMENT 'Nome da estação meteorológica',
    uf STRING COMMENT 'UF da estação',
    regiao STRING COMMENT 'Região do Brasil da estação',
    latitude DOUBLE COMMENT 'Latitude da estação, em grau decimal',
    longitude DOUBLE COMMENT 'Longitude da estação, em grau decimal',
    altitude_m DOUBLE COMMENT 'Altitude da estação, em metros',
    DatCarga TIMESTAMP COMMENT 'Data e hora do processamento na camada trusted'
)
USING DELTA
COMMENT 'Dimensão de estação meteorológica, via mba.raw.clima_inmet.'

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

agregado = (
    spark.table("mba.raw.clima_inmet")
    .select("codigo_wmo", "estacao", "uf", "regiao", "latitude", "longitude", "altitude_m").distinct()
)

# Uma mesma codigo_wmo pode aparecer com pequenas variacoes (nome, uf,
# lat/long) no raw. Ficamos com 1 linha por codigo_wmo para nao dar
# ambiguidade no MERGE (DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE).
w = Window.partitionBy("codigo_wmo").orderBy(F.lit(1))
agregado = (
    agregado
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

agregado.createOrReplaceTempView("stg_clima_diario")
print(f"Linhas agregadas (estacao x dia): {agregado.count():,}")

In [0]:
%sql
MERGE INTO mba.trusted.d_estacao_meterologica AS tgt
USING stg_clima_diario AS src
ON tgt.codigo_wmo = src.codigo_wmo

WHEN MATCHED THEN
    UPDATE SET
        tgt.estacao = src.estacao,
        tgt.uf = src.uf,
        tgt.regiao = src.regiao,
        tgt.latitude = src.latitude,
        tgt.longitude = src.longitude,
        tgt.altitude_m = src.altitude_m,
        tgt.DatCarga = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (
        codigo_wmo, estacao, uf, regiao, latitude, longitude, altitude_m, DatCarga
    )
    VALUES (
        src.codigo_wmo, src.estacao, src.uf, src.regiao, src.latitude, src.longitude, src.altitude_m, current_timestamp()
    );

In [0]:
dbutils.notebook.exit("OK")